In [0]:
from pyspark.sql import functions as F

# 1. 基本統計
print("=== 基本統計 ===")
df_bronze = spark.read.table("bronze_credit_raw")
display(df_bronze.describe())

In [0]:
# 2. Null 值比例
print("=== Null 值比例(%) ===")
total = df_bronze.count()
null_counts = df_bronze.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / total * 100, 2).alias(c)
    for c in df_bronze.columns
])
display(null_counts)

In [0]:
# 3. 異常值
print("=== 各欄位 min/max ===")
display(df_bronze.select(
    F.min("age").alias("age_min"), F.max("age").alias("age_max"),
    F.min("credit_score").alias("cs_min"), F.max("credit_score").alias("cs_max"),
    F.min("debt_ratio").alias("dr_min"), F.max("debt_ratio").alias("dr_max"),
    F.min("income").alias("income_min"), F.max("income").alias("income_max")
))

In [0]:
# 4. default 分佈
print("=== Default 分佈 ===")
display(df_bronze.groupBy("default").count().orderBy("default"))

In [0]:
from pyspark.sql import functions as F

df_silver = df_bronze \
    .filter(F.col("age") > 0) \
    .filter(F.col("debt_ratio") >= 0) \
    .withColumn("credit_score",
        F.when(F.col("credit_score") > 850, None)
         .otherwise(F.col("credit_score"))
    ) \
    .dropna(subset=["income", "loan_amount"])

print(f"Bronze 筆數：{df_bronze.count()}")
print(f"Silver 筆數：{df_silver.count()}")

In [0]:
# 計算中位數
median_cs = df_silver.approxQuantile("credit_score", [0.5], 0.01)[0]
print(f"credit_score 中位數：{median_cs}")

# 用中位數填補 null
df_silver = df_silver.fillna({"credit_score": median_cs})

# 確認沒有 null 了
null_check = df_silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
])
display(null_check)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_credit")

print(f"Silver 寫入完成：{df_silver.count()} 筆")